# Description

In this notebook, I will read the Big Code Bench dataset

In [ ]:
import os 
import time 
import sys
import numpy as np
import pandas as pd
import ast
from datasets import load_dataset
import io
import contextlib

import torch
torch.manual_seed(123)

from utils.model import Llama3Model, generate, text_to_token_ids, token_ids_to_text
from utils.tokenizer import Llama3Tokenizer, ChatFormat, clean_text
from utils.model import LLAMA32_CONFIG_1B, LLAMA32_CONFIG_3B

# 1. Read dataset

In [ ]:
dataset = load_dataset("bigcode/bigcodebench", split='v0.1.4')

print(dataset.column_names, "\n")
print(dataset)

In [ ]:
# Convert to pandas DataFrame
df = pd.DataFrame(dataset)

print(f"Shape of DataFrame: {df.shape}")
df.sample(1)

In [ ]:
idx = np.random.randint(0, len(df))

complete_prompt = df.loc[idx, "complete_prompt"]
canonical_solution = df.loc[idx, "canonical_solution"]
code_prompt = df.loc[idx, "code_prompt"]
test_case = df.loc[idx, "test"]
doc_struct = df.loc[idx, "doc_struct"]
libs = df.loc[idx, "libs"]

print("Complete Prompt:", complete_prompt)
print('-' * 100)
print("\nCanonical Solution:\n", canonical_solution)
print('-' * 100)
print("\nCode Prompt:\n", code_prompt)
print('-' * 100)
print("\nTest Case:\n", test_case)
print('-' * 100)
print("\nDoc Struct:\n", doc_struct)
print('-' * 100)
print("\nLibraries:\n", libs)

In [ ]:
ast.literal_eval(doc_struct)

# 2. Generate code

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")
model.to("cuda")

messages = [
    {"role": "user", "content": "What is the capital of Viet Nam?"},
]
MAX_OUTPUT_TOKENS = 10240

In [ ]:
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=MAX_OUTPUT_TOKENS)
generated_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:])
print("Generated Text:\n", generated_text)

In [ ]:
idx = np.random.randint(0, len(df))

input_prompt = df.loc[idx, "complete_prompt"]
test_case = df.loc[idx, "test"]

print(f"Type of input_prompt: {type(input_prompt)}")
print("Input Prompt:\n", input_prompt)

In [ ]:
# Gernerate code
start = time.time()

messages = [
    {"role": "system", "content": "You are an expert Python engineer. Write concise, correct code."},
    {"role": "user", "content": input_prompt},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
 	pad_token_id=tokenizer.eos_token_id
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=MAX_OUTPUT_TOKENS)

end = time.time()
print(f"Generation Time: {end - start:.2f} seconds")

In [ ]:
generated_code = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:])
print("Generated Code:\n", generated_code)

In [ ]:
# Test case 
full_code = input_prompt + "\n" + generated_code + "\n" + test_case

print("Full Code to Execute:\n", full_code)

In [ ]:
# Capture output safely
f = io.StringIO()
with contextlib.redirect_stdout(f):
    try:
        exec(full_code, {})
        print("✅ Tests passed!")
    except Exception as e:
        print("❌ Test failed:", e)

print(f.getvalue())

# 3. Generate code for whole sample

In [ ]:
out_generated_code = []

for idx in range(len(df)):
    input_prompt = df.loc[idx, "complete_prompt"]
    test_case = df.loc[idx, "test"]

    messages = [
        {"role": "system", "content": "You are an expert Python engineer. Write concise, correct code."},
        {"role": "user", "content": input_prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        pad_token_id=tokenizer.eos_token_id
    ).to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=MAX_OUTPUT_TOKENS)

    generated_code = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:])

    # Test case 
    full_code = input_prompt + "\n" + generated_code + "\n" + test_case

    out_generated_code.append({"input_prompt": input_prompt, "generated_code": generated_code,\
                                "test_case": test_case, "full_code": full_code})


out_generated_code_df = pd.DataFrame(out_generated_code)
out_generated_code_df.to_csv("generated_code_mistral_7B_instruct.csv", index=False)